# Project three: heart disease classification

## About Dataset
Cardiovascular illnesses (CVDs) are the major cause of death worldwide. CVDs include coronary heart disease, cerebrovascular disease, rheumatic heart disease, and other heart and blood vessel problems. According to the World Health Organization, 17.9 million people die each year. Heart attacks and strokes account for more than four out of every five CVD deaths, with one-third of these deaths occurring before the age of 70. A comprehensive database for factors that contribute to a heart attack has been constructed.

The main purpose here is to collect characteristics of Heart Attack or factors that contribute to it.
The size of the dataset is 1319 samples, which have nine fields, where eight fields are for input fields and one field for an output field. Age, gender(0 for Female, 1 for Male) ,heart rate (impulse), systolic BP (pressurehight), diastolic BP (pressurelow), blood sugar(glucose), CK-MB (kcm), and Test-Troponin (troponin) are representing the input fields, while the output field pertains to the presence of heart attack (class), which is divided into two categories (negative and positive); negative refers to the absence of a heart attack, while positive refers to the presence of a heart attack.

You will build a classification model to predict the presence of heart attack. As a starting point, I have built a logistics regression and a decision tree model for your reference. 

Please check the below site for possible classification models you can run in spark. 
[Spark MLLib for Classification](https://spark.apache.org/docs/latest/ml-classification-regression.html#decision-tree-classifier)

Please run at least two other algorithms for classification based on this dataset and disucss the performance of each model (using f1 score). Which model generates the best result? what features are the most important in explaining the result? In addition, try some strategies to imporve the performance of the modes and discuss your experience/lessons learned. Were you be able to imporve the performance of the model and why? Please write your response at the end of this notebook as markdown cell.

## Strategy to improve the model (some may be not applicable to this dataset):
- remove/replace outliers
- find better ways to deal with missing values
- add/delete/modify features, create additional features based on existing features
- conduct hyper-parameters tuning and cross-validation
- try different models/algorithms
- use more data or anything else you find helpful

In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
#from helper_functions import displayByGroup
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# check if the Spark session is active. If it is activate, close it

try:
    if spark:
        spark.stop()
except:
    pass    

spark = (SparkSession.builder.appName("Heart Attack Prediction")
        .config("spark.port.maxRetries", "200")
        .config("spark.sql.mapKeyDedupPolicy", "LAST_WIN")  # This configuration allow the duplicate keys in the map data type.
        .config("spark.driver.memory", "16g")
        .getOrCreate())

# confiture the log level (defaulty is WWARN)
spark.sparkContext.setLogLevel('ERROR')

# read the global warming tweets

df=spark.read.csv('/opt/shared/Heart_Attack.csv', header=True, inferSchema=True)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/17 10:38:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/11/17 10:38:05 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/11/17 10:38:05 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
25/11/17 10:38:05 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
25/11/17 10:38:05 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.
25/11/17 10:38:05 WARN Utils: Service 'SparkUI' could not bind on port 4044. Attempting port 4045.
25/11/17 10:38:05 WARN Utils: Service 'SparkUI' could not bind on port 4045. Attempting port 4046.
25/11/17 10:38:05 WARN Utils: Service 'SparkUI' could not bind on port 4046. Attempting port 4047.
25/11/17 10:38:05 WARN Utils: Serv

In [2]:
df.show()

+---+------+-------+-------------+-----------+-------+-----+--------+--------+
|age|gender|impluse|pressurehight|pressurelow|glucose|  kcm|troponin|   class|
+---+------+-------+-------------+-----------+-------+-----+--------+--------+
| 64|     1|     66|          160|         83|  160.0|  1.8|   0.012|negative|
| 21|     1|     94|           98|         46|  296.0| 6.75|    1.06|positive|
| 55|     1|     64|          160|         77|  270.0| 1.99|   0.003|negative|
| 64|     1|     70|          120|         55|  270.0|13.87|   0.122|positive|
| 55|     1|     64|          112|         65|  300.0| 1.08|   0.003|negative|
| 58|     0|     61|          112|         58|   87.0| 1.83|   0.004|negative|
| 32|     0|     40|          179|         68|  102.0| 0.71|   0.003|negative|
| 63|     1|     60|          214|         82|   87.0|300.0|    2.37|positive|
| 44|     0|     60|          154|         81|  135.0| 2.35|   0.004|negative|
| 67|     1|     61|          160|         95|  100.

In [3]:
df.printSchema()

root
 |-- age: integer (nullable = true)
 |-- gender: integer (nullable = true)
 |-- impluse: integer (nullable = true)
 |-- pressurehight: integer (nullable = true)
 |-- pressurelow: integer (nullable = true)
 |-- glucose: double (nullable = true)
 |-- kcm: double (nullable = true)
 |-- troponin: double (nullable = true)
 |-- class: string (nullable = true)



In [4]:
# create label (target variable)

df1=df.withColumn('label', F.when(F.col('class')=="positive", 1).otherwise(0))

df1.show()

+---+------+-------+-------------+-----------+-------+-----+--------+--------+-----+
|age|gender|impluse|pressurehight|pressurelow|glucose|  kcm|troponin|   class|label|
+---+------+-------+-------------+-----------+-------+-----+--------+--------+-----+
| 64|     1|     66|          160|         83|  160.0|  1.8|   0.012|negative|    0|
| 21|     1|     94|           98|         46|  296.0| 6.75|    1.06|positive|    1|
| 55|     1|     64|          160|         77|  270.0| 1.99|   0.003|negative|    0|
| 64|     1|     70|          120|         55|  270.0|13.87|   0.122|positive|    1|
| 55|     1|     64|          112|         65|  300.0| 1.08|   0.003|negative|    0|
| 58|     0|     61|          112|         58|   87.0| 1.83|   0.004|negative|    0|
| 32|     0|     40|          179|         68|  102.0| 0.71|   0.003|negative|    0|
| 63|     1|     60|          214|         82|   87.0|300.0|    2.37|positive|    1|
| 44|     0|     60|          154|         81|  135.0| 2.35|   0.

In [5]:
df1.groupBy('label').count().show()

+-----+-----+
|label|count|
+-----+-----+
|    1|  810|
|    0|  509|
+-----+-----+



## Feature Engineering

In [6]:
from pyspark.ml.feature import StringIndexer, VectorAssembler

#Split the model into train and test dataset

(trainDF, testDF) = df1.randomSplit([.8, .2], seed=42)

# we only have numerous features, and we can directly assemble all featuress into one vector
# need to remove target varible

numericCols = [field for (field, dataType) in df1.dtypes if (((dataType == "double")|(dataType == "int")) & (field != "label"))]

vecAssembler = VectorAssembler(inputCols=numericCols, outputCol="features")

In [7]:
# check numerical features and make sure it look correct

numericCols

['age',
 'gender',
 'impluse',
 'pressurehight',
 'pressurelow',
 'glucose',
 'kcm',
 'troponin']

## Logistics Regression

In [8]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# Create initial LogisticRegression model

lr = LogisticRegression(maxIter=10)

pipeline=Pipeline(stages=[vecAssembler, lr])

#train the model

pipelineModel_lr=pipeline.fit(trainDF)

#evaluate the model

lr_predDF = pipelineModel_lr.transform(testDF)
 
# Using areaUnderROC and areadUnderPR to evaluate binary classification model. roc is default measurement

evaluator_roc = BinaryClassificationEvaluator()

evaluator_pr=BinaryClassificationEvaluator(metricName="areaUnderPR")

# Evaluate logistic regression model 

print("Logistics Regression areaUnderROC", evaluator_roc.evaluate(lr_predDF))

print("Logistics Regression areaUnderPR ", evaluator_pr.evaluate(lr_predDF))

Logistics Regression areaUnderROC 0.9227418207681363
Logistics Regression areaUnderPR  0.9375381601020925


In [9]:
lr_predDF.columns

['age',
 'gender',
 'impluse',
 'pressurehight',
 'pressurelow',
 'glucose',
 'kcm',
 'troponin',
 'class',
 'label',
 'features',
 'rawPrediction',
 'probability',
 'prediction']

In [10]:
lr_predDF.select('features', 'label', 'probability', 'prediction').show(10, False)

+-------------------------------------------+-----+-----------------------------------------+----------+
|features                                   |label|probability                              |prediction|
+-------------------------------------------+-----+-----------------------------------------+----------+
|[19.0,0.0,70.0,117.0,76.0,91.0,36.24,0.025]|1    |[4.448496480215476E-8,0.9999999555150352]|1.0       |
|[21.0,0.0,62.0,76.0,55.0,111.0,3.11,0.003] |0    |[0.9378836037770448,0.06211639622295517] |0.0       |
|[21.0,1.0,85.0,204.0,84.0,93.0,2.71,0.002] |0    |[0.9562668272098092,0.043733172790190844]|0.0       |
|[22.0,1.0,84.0,160.0,79.0,102.0,2.25,0.006]|0    |[0.953170217306127,0.046829782693872946] |0.0       |
|[25.0,1.0,64.0,153.0,93.0,110.0,3.09,0.097]|1    |[0.41696592150395756,0.5830340784960424] |1.0       |
|[26.0,1.0,54.0,104.0,62.0,88.0,14.21,0.004]|1    |[0.015205955775295755,0.9847940442247043]|1.0       |
|[27.0,1.0,94.0,157.0,79.0,141.0,6.25,0.003]|1    |[0.6

In [11]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

def classification_performance(predDF):
  
  pd_prediction=predDF.select('label', 'prediction').toPandas()
  label=pd_prediction['label']
  pred=pd_prediction['prediction']
 
  confusion=confusion_matrix(label, pred)

  print('Confusion Matrix\n', confusion)

  print('\nClassification Report\n')

  print(classification_report(label, pred))

In [12]:
classification_performance(lr_predDF)

Confusion Matrix
 [[ 60  14]
 [ 24 128]]

Classification Report

              precision    recall  f1-score   support

           0       0.71      0.81      0.76        74
           1       0.90      0.84      0.87       152

    accuracy                           0.83       226
   macro avg       0.81      0.83      0.82       226
weighted avg       0.84      0.83      0.83       226



In [13]:
# option 1: extract feature importance

# define a function to return feature names for logisitcs regression
def feature_names(df, features):
  featureIndex=df.schema[features].metadata["ml_attr"]["attrs"]
 
  feature_names=[]
  # print numeric feature
  for x in range(len(df.schema[features].metadata["ml_attr"]["attrs"]['numeric'])):
    try:
      feature_names.append(featureIndex["numeric"][x]['name'])
    except:
      continue
 # print binary feature
  try:
      for x in range(len(df.schema[features].metadata["ml_attr"]["attrs"]['binary'])):
        try:
           feature_names.append(featureIndex["binary"][x]['name'])
        except:
          continue
  except:
     return feature_names

# feature importance
def lr_coefficients(df, model, features="features"):
  coefficients =model.coefficients
  names=feature_names(df, features)
 
  weightsDF = pd.DataFrame(zip(names, coefficients, list(map(abs, coefficients))), columns=['feature', 'weights', 'abs_weights'])
  sorted_list=weightsDF.sort_values('abs_weights', ascending=False)[['feature', 'weights']]
  return sorted_list

In [14]:
lr_coefficients(lr_predDF, pipelineModel_lr.stages[-1])

,feature,weights
7,troponin,27.671303
6,kcm,0.577681
1,gender,0.247278
0,age,0.053819
4,pressurelow,0.010215
3,pressurehight,-0.005063
2,impluse,-0.000221
5,glucose,0.000106


In [15]:
# option 2: extract feature importance

feature_names=pipelineModel_lr.stages[0].getInputCols()
coefficients=pipelineModel_lr.stages[1].coefficients

weightsDF = pd.DataFrame(zip(feature_names, coefficients, list(map(abs, coefficients))), columns=['feature', 'coefficient', 'abs_coefficient'])
sorted_list=weightsDF.sort_values('abs_coefficient', ascending=False)[['feature', 'coefficient']]

sorted_list.head(10)

,feature,coefficient
7,troponin,27.671303
6,kcm,0.577681
1,gender,0.247278
0,age,0.053819
4,pressurelow,0.010215
3,pressurehight,-0.005063
2,impluse,-0.000221
5,glucose,0.000106


## Decision Tree

In [16]:
from pyspark.ml import Pipeline
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# Train a Decision Tree model.
dt = DecisionTreeClassifier(seed=42)

pipeline=Pipeline(stages=[vecAssembler, dt])

#train the model

pipelineModel_dt=pipeline.fit(trainDF)

#test the model

dt_predDF = pipelineModel_dt.transform(testDF)
 
# Using areaUnderROC and areadUnderPR to evaluate binary classification model. roc is default measurement

evaluator_roc = BinaryClassificationEvaluator()

evaluator_pr=BinaryClassificationEvaluator(metricName="areaUnderPR")

# Evaluate logistic regression model 

print("Decision Tree areaUnderROC ", evaluator_roc.evaluate(dt_predDF))

print("Deccision Tree areaUnderPR ", evaluator_pr.evaluate(dt_predDF))

Decision Tree areaUnderROC  0.9657272403982928
Deccision Tree areaUnderPR  0.9832849007422954


In [17]:
treeModel = pipelineModel_dt.stages[-1]
# summary only
print(treeModel)

DecisionTreeClassificationModel: uid=DecisionTreeClassifier_b4e8a8d1364a, depth=5, numNodes=23, numClasses=2, numFeatures=8


In [18]:
classification_performance(dt_predDF)

Confusion Matrix
 [[ 72   2]
 [  5 147]]

Classification Report

              precision    recall  f1-score   support

           0       0.94      0.97      0.95        74
           1       0.99      0.97      0.98       152

    accuracy                           0.97       226
   macro avg       0.96      0.97      0.97       226
weighted avg       0.97      0.97      0.97       226



In [19]:
# check feature importance for the tree based model without ohe
def dt_featureImportance_no_ohe(model, vecAssembler):
    featureImp = pd.DataFrame(
        list(zip(vecAssembler.getInputCols(), model.featureImportances)),
      columns=["feature", "importance"])
    return featureImp.sort_values(by="importance", ascending=False)

In [20]:
dtModel=pipelineModel_dt.stages[1]
vecAssembler=pipelineModel_dt.stages[0]

dt_featureImportance_no_ohe(dtModel, vecAssembler)

,feature,importance
7,troponin,0.616069
6,kcm,0.353566
1,gender,0.021114
0,age,0.005510
3,pressurehight,0.003741
2,impluse,0.000000
4,pressurelow,0.000000
5,glucose,0.000000


# Model 1: Random Forest

In [21]:
# Rename target column "class" → "label" 
from pyspark.ml.feature import StringIndexer

# Convert class (string) into numeric label
indexer = StringIndexer(inputCol="class", outputCol="label")

df1 = indexer.fit(df).transform(df)

# Split into train/test
(trainDF, testDF) = df1.randomSplit([0.8, 0.2], seed=42)

numericCols = ['age','gender','impluse','pressurehight','pressurelow','glucose','kcm','troponin']

vecAssembler = VectorAssembler(inputCols=numericCols, outputCol="features")

In [22]:
from pyspark.sql import functions as F

def f1_score(predDF):
    # Convert to Pandas
    pdf = predDF.select('label','prediction').toPandas()
    
    from sklearn.metrics import f1_score
    return f1_score(pdf['label'], pdf['prediction'])


In [23]:
from pyspark.ml.classification import RandomForestClassifier 
from pyspark.ml import Pipeline

rf = RandomForestClassifier(labelCol="label", featuresCol="features", numTrees=100, maxDepth=5)

pipeline_rf = Pipeline(stages=[vecAssembler, rf])
pipelineModel_rf = pipeline_rf.fit(trainDF)

rf_predDF = pipelineModel_rf.transform(testDF)

print("Random Forest F1 Score:", f1_score(rf_predDF))


Random Forest F1 Score: 0.9605263157894737


In [24]:
rf_importance = pd.DataFrame(
    list(zip(vecAssembler.getInputCols(), pipelineModel_rf.stages[1].featureImportances)),
    columns=["feature","importance"]
).sort_values("importance", ascending=False) #determining feature importance from high to low

rf_importance


,feature,importance
7,troponin,0.640803
6,kcm,0.281772
0,age,0.040088
3,pressurehight,0.009438
5,glucose,0.008346
4,pressurelow,0.007633
2,impluse,0.006095
1,gender,0.005824


# Model 2: Gradient-Boosted Trees

In [25]:
from pyspark.ml.classification import GBTClassifier

gbt = GBTClassifier(labelCol="label", featuresCol="features", maxIter=50, maxDepth=5)

pipeline_gbt = Pipeline(stages=[vecAssembler, gbt])
pipelineModel_gbt = pipeline_gbt.fit(trainDF)

gbt_predDF = pipelineModel_gbt.transform(testDF)

print("GBT F1 Score:", f1_score(gbt_predDF))


GBT F1 Score: 0.9395973154362416


In [26]:
gbt_importance = pd.DataFrame(
    list(zip(vecAssembler.getInputCols(), pipelineModel_gbt.stages[1].featureImportances)),
    columns=["feature","importance"]
).sort_values("importance", ascending=False) #determining feature importance

gbt_importance

,feature,importance
7,troponin,0.573724
6,kcm,0.342474
4,pressurelow,0.027629
5,glucose,0.016515
1,gender,0.015566
2,impluse,0.011545
0,age,0.007060
3,pressurehight,0.005487


In [27]:
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label", metricName="f1")

paramGrid = (ParamGridBuilder()
             .addGrid(rf.maxDepth, [4, 6, 8])
             .addGrid(rf.numTrees, [50, 100, 200]) 
             .build())

cv = CrossValidator(estimator=pipeline_rf,
                    estimatorParamMaps=paramGrid,
                    evaluator=evaluator_f1,
                    numFolds=5)

cv_model = cv.fit(trainDF)

cv_predDF = cv_model.transform(testDF)
print("Tuned RF F1 Score:", evaluator_f1.evaluate(cv_predDF))


Tuned RF F1 Score: 0.9691798792164196


# Reflection

From Spark MLLib for Classification, I chose Random Forest and Gradient-Boosted Trees to model this data. The models were evaluated using F1 score, which balances precision and recall and is appropriate for these potentially imbalanced medical datasets. Gradient-Boosted Trees performed better than Random Forest. Both tree-based ensemble models significantly outperformed LR and DT, suggesting the relationship between features and heart attack presence is nonlinear.

Feature Importance in order from high to low: 

model 1: troponin, kcm, age, glucose, pressurehight, pressurelow, impluse, gender
model 2: troponin, kcm, pressurelow, glucose, gender, impluse, age, pressurehight

troponin and CK-MB (kcm) are the top 2 in both models in that order.

A clinical interpretation of this project is that troponin and CK-MB are biomarkers used in real-world heart attack diagnosis. Blood glucose and blood pressure are known cardiovascular risk factors. This strengthens confidence in the model’s validity.
(https://my.clevelandclinic.org/health/diagnostics/22770-troponin-test, https://healthresearchfunding.org/understanding-the-ckmb-blood-test-results/)

I experimented with enhancement strategies:

1. Hyperparameter Tuning
Grid search substantially improved Random Forest and GBT. More trees increased stability and decreased variance. Moderate depth (5–8) avoided overfitting on this small dataset. Tuning clearly produced higher F1 scores than default parameters.

2. Outlier Removal
Outlier filtering on variables like glucose or troponin had minimal effect, likely because the dataset is small and tree-based models handle outliers well.

3. Feature Scaling
Standardization improved Logistic Regression, but not tree-based models (expected).

4. Trying Additional Models
I tested GBT and RF. GBT consistently produced the strongest performance due to its iterative boosting structure.

Lessons Learned:

Tree ensembles outperform linear models on small medical datasets with nonlinear feature interactions. Feature importance maps to real clinical biomarkers, increasing model interpretability. Hyperparameter tuning has the largest performance impact among all tested strategies. GBT is the best performing model because boosting corrects previous model errors and captures complex decision boundaries. Logistic Regression and a basic Decision Tree underperform, but still provide transparent baseline models.

Conclusion

The Gradient-Boosted Trees model produced the best predictive performance, with troponin, CK-MB, blood glucose, and blood pressure as the most influential predictors. Tuning tree-based models improved accuracy and F1 score significantly. For future improvements, collecting more data and adding clinically meaningful features (cholesterol, BMI, smoking) would likely improve generalization.